In [ ]:
import numpy as np
import numpy.typing as npt
import networkx as nx
import matplotlib.pyplot as plt
import casadi as ca

In [ ]:
def headloss_pipes(
    inductance: float | npt.NDArray,
    resistance: float | npt.NDArray,
    volume_flow: float | npt.NDArray,
) -> float | npt.NDArray:
    if not (inductance.shape == resistance.shape == volume_flow.shape):
        raise ValueError("Input values not of the same shape.")
    with np.errstate(divide="raise", invalid="raise"):
        return (resistance / inductance) * abs(volume_flow) * volume_flow

In [ ]:
## constants
# gravitational constant
g = 9.81  # m/s^2
# wave speed
a = 1.0e3  # m/s
# density
rho = 998.2  # kg/m^3
# dynamic viscosity
mu = 1.002e-3  # Ns/m^2

In [ ]:
## average operating conditions
# volume flow
q_mean = 0.0001  # 0224331792911098  # m^3/2
# pump speed
omega_mean = 36.410179219931635  # 35.35520636739956  # 9.7996642  # Hz
omega_max = 74.166667
omega_mean_relative = omega_mean / 74.166667  # dimensionless
mean_pump_pressure = 25274.0257666655

In [ ]:
internal_nodes = {
    "pump_inlet": {
        "name": "pump_inlet",
        "elevation": 0.33,
        "demand": 0,
    },
    "pump_outlet": {
        "name": "pump_outlet",
        "elevation": 0.47,
        "demand": 0,
    },
    "tank_inlet": {
        "name": "tank_inlet",
        "elevation": 2.12,
        "demand": 0.0001,  # 0224331792911098
    },
}

In [ ]:
reservoirs = {
    "reservoir_outlet": {
        "name": "reservoir_outlet",
        "elevation": 0.56,
        "pressure": 0.3833,
    },
}

In [ ]:
pipes = {
    "pump_suction_pipe": {
        "start_node": "reservoir_outlet",
        "end_node": "pump_inlet",
        "type": "pipe",
        "length": 2.25,
        "diameter": 0.015,
        "roughness": 0.007e-3,
        "friction_factor": 3.383e-3,  # 2.65e-1,
    },
    "tank_feeding_pipe": {
        "start_node": "pump_outlet",
        "end_node": "tank_inlet",
        "type": "pipe",
        "length": 5,
        "diameter": 0.015,
        "roughness": 0.007e-3,
        "friction_factor": 4.4e-3,  # 1.1357142e-1,
    },
}

In [ ]:
elements = {
    "pump": {
        "start_node": "pump_inlet",
        "end_node": "pump_outlet",
        "type": "pump",
        "diameter": 0.015,
        "length": 0.1,
        "head_coefficients": [-0.1045223, 0.18435539, 10.63028341],
        "power_coefficients": [
            -3.24482049,
            7.30651404,
            20.33482845,
            103.00866719,
            7.59352180888483,
        ],
        "system_coefficent": 0.3,  # 0.2,
        "min_speed": 23.33333,
        "max_speed": 74.16667,
        "initial_input": omega_mean,  # 35.35520636739956,
    },
}

In [ ]:
pipe_number = len(pipes.keys())
internal_node_number = len(internal_nodes.keys())
reservoir_number = len(reservoirs.keys())
element_number = len(elements.keys())

In [ ]:
for name, properties in elements.items():
    properties["cross_section_area"] = (properties["diameter"] / 2) ** 2 * np.pi
a_0 = elements["pump"]["cross_section_area"]
pump_coeffs = elements["pump"]["head_coefficients"]

In [ ]:
for name, properties in pipes.items():
    properties["cross_section_area"] = (properties["diameter"] / 2) ** 2 * np.pi

In [ ]:
inverse_relative_roughnesses = {}
for name, properties in pipes.items():
    inverse_relative_roughnesses[name] = (
        properties["diameter"] / properties["roughness"]
    )

In [ ]:
initial_condition_flow_pipes = np.full((len(pipes.keys()), 1), q_mean)
initial_condition_flow_elements = np.full((len(elements.keys()), 1), q_mean)

In [ ]:
edge_dict = {}
for name, edge in pipes.items():
    edge_dict[name] = (edge["start_node"], edge["end_node"])
for name, edge in elements.items():
    edge_dict[name] = (edge["start_node"], edge["end_node"])
graph_edge_list = [edge for edge in edge_dict.values()]

In [ ]:
graph = nx.Graph()
graph.add_nodes_from(reservoirs.keys())
graph.add_nodes_from(internal_nodes.keys())
graph.add_edges_from(edge_dict.values())

In [ ]:
internal_nodes_list = [name for name in internal_nodes.keys()]
reservoir_nodes_list = [name for name in reservoirs.keys()]
node_list = internal_nodes_list + reservoir_nodes_list

In [ ]:
pipe_list = [name for name in pipes.keys()]
element_list = [name for name in elements.keys()]
edge_list = pipe_list + element_list

In [ ]:
incidence_matrix = -nx.incidence_matrix(
    graph, oriented=True, nodelist=node_list, edgelist=graph_edge_list
)
print(incidence_matrix.todense())

In [ ]:
A_I = np.array(incidence_matrix.todense()[:internal_node_number])
A_R = np.array(incidence_matrix.todense()[-reservoir_number:])

In [ ]:
A_I_p = A_I[:, :pipe_number]
A_I_e = A_I[:, -element_number:]
A_R_p = A_R[:, :pipe_number]
A_R_e = A_R[:, -element_number:]

In [ ]:
inductance = []
resistance = []
capacitance = []
for properties in pipes.values():
    inductance.append(properties["length"] / (g * properties["cross_section_area"]))
    resistance.append(
        8
        * properties["length"]
        * properties["roughness"]
        / (np.pi**2 * g * properties["diameter"] ** 5)
    )
    capacitance.append(
        (2 * g * np.pi / 4 * properties["diameter"] ** 2 * properties["length"])
        / (a**2)
    )
for properties in elements.values():
    capacitance.append(
        (2 * g * np.pi / 4 * properties["diameter"] ** 2 * properties["length"])
        / (a**2)
    )

In [ ]:
system_coeffcients = []
for properties in elements.values():
    system_coeffcients.append(properties["system_coefficent"])

In [ ]:
L = np.array(inductance).reshape(len(list(pipes.keys())), 1)
R = np.array(resistance).reshape(len(list(pipes.keys())), 1)
C = np.array(capacitance).reshape(len(list(edge_dict.keys())), 1)

In [ ]:
f_p_initial = headloss_pipes(L, R, initial_condition_flow_pipes)
f_p_initial

In [ ]:
reservoir_total_head = (
    reservoirs["reservoir_outlet"]["pressure"]
    + (reservoirs["reservoir_outlet"]["elevation"])
)
reservoir_total_head

In [ ]:
internal_nodes["pump_inlet"]["initial_head"] = (
    0.9432828625777657  #  1.39962220  # (internal_nodes["consumer_valve_inlet"]["elevation"])  # internal_nodes["consumer_valve_outlet"]["initial_head"] + R[1,0] * q_mean**2 +
)
internal_nodes["pump_outlet"]["initial_head"] = (
    (25274.0257666655 / (rho * g)) + internal_nodes["pump_inlet"]["initial_head"]
)  # internal_nodes["pump_outlet"]["elevation"] #  1.39800676  # (internal_nodes["consumer_valve_outlet"]["elevation"])  # internal_nodes["consumer_outlet"]["initial_head"] + R[2,0] * q_mean**2 +
internal_nodes["tank_inlet"]["initial_head"] = (
    3.524243866935379
    - 0.2  #  1.39799373  # internal_nodes["consumer_outlet"]["elevation"]  # (1/(2*g))*(q_mean/pipes["consumer_outlet_pipe"]["cross_section_area"])**2
)

In [ ]:
(
    internal_nodes["pump_outlet"]["initial_head"]
    - internal_nodes["pump_inlet"]["initial_head"]
)

In [ ]:
for name, properties in pipes.items():
    properties["friction_factor"] = (
        (
            internal_nodes[properties["start_node"]]["initial_head"]
            - internal_nodes[properties["end_node"]]["initial_head"]
        )
        / (
            (properties["length"] / properties["diameter"])
            * (1 / (2 * g))
            * (q_mean / properties["cross_section_area"]) ** 2
        )
        if properties["start_node"] in internal_nodes.keys()
        else (
            reservoirs["reservoir_outlet"]["pressure"]
            + (reservoirs["reservoir_outlet"]["elevation"])
            - internal_nodes[properties["end_node"]]["initial_head"]
        )
        / (
            (properties["length"] / properties["diameter"])
            * (1 / (2 * g))
            * (q_mean / properties["cross_section_area"]) ** 2
        )
    )
    print(f"{name}: {properties['friction_factor']}")

In [ ]:
h_I_0 = np.array([node["initial_head"] for node in internal_nodes.values()]).reshape(
    internal_node_number, 1
)
h_I_0

In [ ]:
h_R_0 = np.array([reservoir_total_head]).reshape(reservoir_number, 1)

In [ ]:
Q_0 = np.array([node["demand"] for node in internal_nodes.values()]).reshape(
    internal_node_number, 1
)

In [ ]:
z_0 = np.array([elements["pump"]["initial_input"]]).reshape(element_number, 1)

In [ ]:
u_0 = np.concatenate((h_R_0, Q_0, z_0))

In [ ]:
G = np.linalg.inv(np.diagflat((0.5 * (abs(A_I) @ C))))

In [ ]:
D = np.diagflat(np.array(system_coeffcients).reshape(element_number, element_number))

In [ ]:
A = np.block(
    [
        [
            np.zeros(shape=(pipe_number, pipe_number)),
            np.zeros(shape=(pipe_number, element_number)),
            np.linalg.inv(np.diagflat(L)) @ A_I_p.transpose(),
            np.zeros(shape=(pipe_number, element_number)),
        ],
        [
            np.zeros(shape=(element_number, pipe_number)),
            np.zeros(shape=(element_number, element_number)),
            A_I_e.transpose(),
            np.zeros(shape=(element_number, element_number)),
        ],
        [
            -G @ A_I_p,
            -G @ A_I_e,
            np.zeros(shape=(internal_node_number, internal_node_number)),
            np.zeros(shape=(internal_node_number, element_number)),
        ],
        [
            np.zeros(shape=(element_number, pipe_number)),
            np.zeros(shape=(element_number, element_number)),
            np.zeros(shape=(element_number, internal_node_number)),
            -D,
        ],
    ]
)

In [ ]:
B = np.block(
    [
        [
            np.linalg.inv(np.diagflat(L)) @ A_R_p.transpose(),
            np.zeros(shape=(pipe_number, internal_node_number)),
            np.zeros(shape=(pipe_number, element_number)),
        ],
        [
            A_R_e.transpose(),
            np.zeros(shape=(element_number, internal_node_number)),
            np.zeros(shape=(element_number, element_number)),
        ],
        [
            np.zeros(shape=(internal_node_number, reservoir_number)),
            -G,
            np.zeros(shape=(internal_node_number, element_number)),
        ],
        [
            np.zeros(shape=(element_number, reservoir_number)),
            np.zeros(shape=(element_number, internal_node_number)),
            D,
        ],
    ]
)

In [ ]:
F_p_0 = (
    np.linalg.inv(np.diagflat(L))
    @ np.diagflat(R)
    @ np.full((1, pipe_number), np.abs(q_mean) * q_mean)[0]
).reshape(pipe_number, 1)

In [ ]:
(
    pump_coeffs[0] * (q_mean * 3600) ** 2
    + pump_coeffs[1] * (q_mean * 3600) * (omega_mean / omega_max)
    + pump_coeffs[2] * (omega_mean / omega_max) ** 2
    - (h_I_0[1, 0] - h_I_0[0, 0])
)

In [ ]:
omega_init = np.roots(
    [
        pump_coeffs[2] * (1 / omega_max) ** 2,
        pump_coeffs[1] * (q_mean * 3600) * (1 / omega_max),
        pump_coeffs[0] * (q_mean * 3600) ** 2 - (h_I_0[1, 0] - h_I_0[0, 0]),
    ]
)

In [ ]:
omega_init[1]

In [ ]:
pump_coeffs_si = [
    1 * pump_coeffs[0] * (3600**2),
    1 * pump_coeffs[1] * (3600 / omega_max),
    pump_coeffs[2] / (omega_max**2),
]

In [ ]:
(
    pump_coeffs_si[0] * (q_mean) ** 2
    + pump_coeffs_si[1] * (q_mean) * (omega_mean)
    + pump_coeffs_si[2] * (omega_mean) ** 2
    - (h_I_0[1, 0] - h_I_0[0, 0])
)

In [ ]:
(
    pump_coeffs_si[0] * (q_mean) ** 2
    + pump_coeffs_si[1] * (q_mean) * (omega_mean)
    + pump_coeffs_si[2] * (omega_mean) ** 2
)

In [ ]:
omega_mean

In [ ]:
pump_coeffs_si

In [ ]:
def f_pump_curve(q, omega, pump_coeffs):
    return (
        1 * pump_coeffs[0] * (q) ** 2
        + 1 * pump_coeffs[1] * (q) * (omega)
        + 1 * pump_coeffs[2] * (omega) ** 2
        # +2
    )


q_list = np.linspace(0.8 * q_mean, 2 * q_mean, 100)

plt.plot(q_list, [f_pump_curve(q, omega_mean, pump_coeffs_si) for q in q_list])
plt.xlabel("Q")
plt.ylabel("H")

In [ ]:
F_p_0 = (
    np.linalg.inv(np.diagflat(L))
    @ np.diagflat(R)
    @ np.full((1, pipe_number), np.abs(q_mean) * q_mean)[0]
).reshape(pipe_number, 1)
F_p_0

In [ ]:
F_e_0 = -(  # -
    pump_coeffs_si[0] * (q_mean) ** 2
    + pump_coeffs_si[1] * (q_mean) * (omega_mean)
    + pump_coeffs_si[2] * (omega_mean) ** 2
    # 2
)
F_e_0

In [ ]:
F_0 = np.block(
    [
        [F_p_0],
        [F_e_0],
        [np.zeros(shape=(internal_node_number, 1))],
        [np.zeros(shape=(element_number, 1))],
    ]
)
F_0

In [ ]:
x_0 = np.block(
    [
        [q_mean * np.ones((pipe_number, 1))],
        [q_mean * np.ones((element_number, 1))],
        [h_I_0],
        [z_0],
    ]
)
x_0

In [ ]:
(x_0[4] - x_0[3])[0]

In [ ]:
E = np.block(
    [
        [
            np.eye(pipe_number),
            np.zeros((pipe_number, element_number)),
            np.zeros((pipe_number, internal_node_number)),
            np.zeros((pipe_number, element_number)),
        ],
        [
            np.zeros((element_number, pipe_number)),
            np.zeros((element_number, element_number)),
            np.zeros((element_number, internal_node_number)),
            np.zeros((element_number, element_number)),
        ],
        [
            np.zeros((internal_node_number, pipe_number)),
            np.zeros((internal_node_number, element_number)),
            np.eye(internal_node_number),
            np.zeros((internal_node_number, element_number)),
        ],
        [
            np.zeros((element_number, pipe_number)),
            np.zeros((element_number, element_number)),
            np.zeros((element_number, internal_node_number)),
            np.eye(element_number),
        ],
    ]
)

In [ ]:
alg_vars = np.isclose(np.linalg.norm(E, axis=1), 0.0)

In [ ]:
rhs_0 = A @ x_0 - F_0 + B @ u_0
x_dot_0 = np.zeros_like(x_0)
diff_rows = ~alg_vars
E_mod = E[diff_rows]
E_dyn = E_mod[:, diff_rows]
x_dot_0[diff_rows] = np.linalg.solve(E_dyn, rhs_0[diff_rows])
x_dot_0[alg_vars] = 0.0

In [ ]:
rhs_0

In [ ]:
A_mod = A[diff_rows]
A_dyn = A_mod[:, diff_rows]
B_dyn = B[diff_rows]

#### Variables

In [ ]:
# Dynamic state
flow_pump_suction_pipe = ca.MX.sym("flow_pump_suction_pipe")
flow_tank_feeding_pipe = ca.MX.sym("flow_tank_feeding_pipe")
head_pump_inlet = ca.MX.sym("head_pump_inlet")
head_pump_outlet = ca.MX.sym("head_pump_outlet")
head_tank_inlet = ca.MX.sym("head_tank_inlet")
pump_speed = ca.MX.sym("pump_speed")
x = ca.vertcat(
    flow_pump_suction_pipe,
    flow_tank_feeding_pipe,
    head_pump_inlet,
    head_pump_outlet,
    head_tank_inlet,
    pump_speed,
)

x_names = [
    "flow_pump_suction_pipe",
    "flow_tank_feeding_pipe",
    "head_pump_inlet",
    "head_pump_outlet",
    "head_tank_inlet",
    "pump_speed",
]

# x_dot_sym = ca.MX.sym("x_dot_sym", x.shape[0])
# x_dot = ca.vertcat(x_dot_sym)

In [ ]:
lb_flow = np.full((pipe_number), 0)
ub_flow = np.full((pipe_number), 5 * q_mean)
lb_head = np.full((internal_node_number), 0)
ub_head = np.full((internal_node_number), 5 * reservoir_total_head)  # orig: 1*
lb_pump = np.full((element_number), 0)  # orig: 0
ub_pump = np.full((element_number), 75)  # orig: 1
lbx = ca.vertcat(*lb_flow, *lb_head, *lb_pump)
ubx = ca.vertcat(*ub_flow, *ub_head, *ub_pump)

In [ ]:
# Algebraic state
flow_pump = ca.MX.sym("flow_pump")
alg_state = ca.vertcat(flow_pump)
lb_flow_pump = np.full(element_number, 0)
ub_flow_pump = np.full(element_number, 5 * q_mean)
lb_alg_state = ca.vertcat(lb_flow_pump)
up_alg_state = ca.vertcat(ub_flow_pump)

In [ ]:
# Control inputs
reservoir_head = ca.MX.sym("reservoir_head")
demand_pump_inlet = ca.MX.sym("demand_pump_inlet")
demand_pump_outlet = ca.MX.sym("demand_pump_outlet")
demand_tank_inlet = ca.MX.sym("demand_tank_inlet")
pump_input = ca.MX.sym("pump_input")
u = ca.vertcat(
    reservoir_head, demand_pump_inlet, demand_pump_outlet, demand_tank_inlet, pump_input
)

In [ ]:
lb_reservoir_head = np.full(reservoir_number, 0.8 * reservoir_total_head)
lb_demand = np.full(internal_node_number, 0)
lb_pump_input = np.full(element_number, 0)
lbu = ca.vertcat(*lb_reservoir_head, *lb_demand, *lb_pump_input)

ub_reservoir_head = np.full(reservoir_number, 1.2 * reservoir_total_head)
ub_demand = np.full(internal_node_number, 5 * q_mean)
ub_pump_input = np.full(element_number, 74)
ubu = ca.vertcat(*ub_reservoir_head, *ub_demand, *ub_pump_input)

In [ ]:
pipe_flows = ca.vertcat(flow_pump_suction_pipe, flow_tank_feeding_pipe)

In [ ]:
L_inv_times_R = np.linalg.inv(np.diagflat(L)) @ np.diagflat(R)

In [ ]:
# Non-linear termns
F_p_flow_pump_suction_pipe = (
    L_inv_times_R[0, 0] * ca.fabs(flow_pump_suction_pipe) * flow_pump_suction_pipe
)
F_p_flow_tank_feeding_pipe = (
    L_inv_times_R[1, 1] * ca.fabs(flow_tank_feeding_pipe) * flow_tank_feeding_pipe
)
F_q_pump = G @ A_I_e @ alg_state  # orig: plus
F_p = ca.vertcat(
    F_p_flow_pump_suction_pipe,
    F_p_flow_tank_feeding_pipe,
    F_q_pump,
    *([0] * element_number),
)

In [ ]:
(
    pump_coeffs_si[0] * (q_mean) ** 2
    + pump_coeffs_si[1] * (q_mean) * (omega_mean)
    + pump_coeffs_si[2] * (omega_mean) ** 2
)

In [ ]:
# Equations
alg_eq = -(  # -
    pump_coeffs_si[0] * (flow_pump) ** 2
    + pump_coeffs_si[1] * (flow_pump) * (pump_speed)
    + pump_coeffs_si[2] * (pump_speed) ** 2
    # 2
) + (head_pump_outlet - head_pump_inlet)  # orig: +()

In [ ]:
x_dot = A_dyn @ x + B_dyn @ u - F_p
x_dot

In [ ]:
dt = 1e-4
t = np.arange(0, 50, dt)
t0 = 0

dae = {"x": x, "z": alg_state, "p": u, "ode": x_dot, "alg": alg_eq}

opts = {
    "tf": 1e-4,  # time step for each integrator call
    #     "abstol": 1e-8,
    #     "reltol": 1e-8,
}

Fint = ca.integrator("Fint", "idas", dae, t0, dt, {"calc_ic": True})  #

# initial conditions
x0 = ca.DM(x_0[diff_rows])
z0 = ca.DM(x_0[alg_vars])

u0 = ca.DM(u_0)
u0 = []
tc = 15
LL = 1  # logistic function coeff
k = 10
for tt in t:
    # if tt < tc:
    #     u0.append(ca.DM(u_0))
    # else:
    #     u0.append(ca.DM((u_0 * np.array([1, 1, 1, 1, 1]).reshape(5, 1)).flatten()))
    u0.append(
        ca.DM(
            (
                u_0
                * np.array(
                    [1, 1, 1, 1, 1 + 0.4 * LL / (1 + np.e ** (-k * (tt - tc)))]
                ).reshape(5, 1)
            ).flatten()
        )
    )


x_vals = np.zeros((len(t), x0.numel()))
x_vals[0, :] = np.array(x0.T).ravel()
xt = x0

z_vals = np.zeros((len(t), z0.numel()))
z_vals[0, :] = np.array(z0.T).ravel()
zt = z0

res = Fint(x0=xt, z0=zt, p=u0[0])

In [ ]:
for i, tt in enumerate(t):
    res = Fint(x0=xt, z0=zt, p=u0[i])
    xt = res["xf"]
    zt = res["zf"]
    x_vals[i, :] = np.array(xt.T).ravel()
    z_vals[i, :] = np.array(zt.T).ravel()

In [ ]:
fig, axs = plt.subplots(len(x_0) - 1, figsize=(5, 20))

for i in range(len(x_0) - 1):
    axs[i].plot(
        t[::1000], x_vals[::1000, i], label=x_names[i]
    )  # plotting every 1000th point
    axs[i].legend()

# axs[-1].set_ylim((36, 36.5))

In [ ]:
len(t)

In [ ]:
len(x_vals)

In [ ]:
np.mean(x_vals[:, 2])

In [ ]:
x_vals[0, 4]

In [ ]:
x_vals[0, 3]

In [ ]:
fig, axs = plt.subplots(len(x_0[alg_vars]), figsize=(5, 5))

for i in range(len(x_0[alg_vars])):
    axs.plot(t, z_vals[:, i])

In [ ]:
ttt = np.arange(0, 25, 1e-4)

L = 1
k = 1
t0 = 10
plt.plot(ttt, [L / (1 + np.e ** (-k * (t - t0))) for t in ttt])

In [ ]:
from scipy.integrate import solve_ivp

In [ ]:
# Parameters
a = 0.3
t_eval = np.arange(0, 50, dt)


k = 20


# v(t) piecewise definition
def v_of_t(t):
    return 5 * (1 + 0.3 * LL / (1 + np.e ** (-k * (t - tc))))


# ODE
def dsdt(t, s):
    return a * (v_of_t(t) - s)


# Initial condition
s0 = [5.0]

# Solve
sol = solve_ivp(dsdt, (t_eval[0], t_eval[-1]), s0, t_eval=t_eval)

# Plot
plt.plot(sol.t, sol.y[0], label="s(t)")
plt.axvline(10, color="gray", linestyle="--", alpha=0.6)
plt.xlabel("t")
plt.ylabel("s")
plt.legend()
plt.show()